In [27]:
import pandas as pd
import numpy as np
import yfinance as yf

In [28]:
nifty = yf.download("^NSEI", period="5y")

nifty.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2021-06-07,15751.650391,15773.450195,15678.099609,15725.099609,394000
2021-06-08,15740.099609,15778.799805,15680.000000,15773.900391,378200
2021-06-09,15635.349609,15800.450195,15566.900391,15766.299805,457900
2021-06-10,15737.750000,15751.250000,15648.500000,15692.099609,298300
2021-06-11,15799.349609,15835.549805,15749.799805,15796.450195,363000


In [29]:
print(nifty.shape)

nifty.info()

(1235, 5)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1235 entries, 2021-06-07 to 2026-06-05
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, ^NSEI)   1235 non-null   float64
 1   (High, ^NSEI)    1235 non-null   float64
 2   (Low, ^NSEI)     1235 non-null   float64
 3   (Open, ^NSEI)    1235 non-null   float64
 4   (Volume, ^NSEI)  1235 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 57.9 KB


In [30]:
nifty = nifty.reset_index()

nifty.head()

Price,Date,Close,High,Low,Open,Volume
Ticker,,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
0,2021-06-07,15751.650391,15773.450195,15678.099609,15725.099609,394000
1,2021-06-08,15740.099609,15778.799805,15680.000000,15773.900391,378200
2,2021-06-09,15635.349609,15800.450195,15566.900391,15766.299805,457900
3,2021-06-10,15737.750000,15751.250000,15648.500000,15692.099609,298300
4,2021-06-11,15799.349609,15835.549805,15749.799805,15796.450195,363000


In [31]:
nifty["Daily_Return"] = nifty["Close"].pct_change() * 100

nifty.head()

Price,Date,Close,High,Low,Open,Volume,Daily_Return
Ticker,,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,
0,2021-06-07,15751.650391,15773.450195,15678.099609,15725.099609,394000,NaN
1,2021-06-08,15740.099609,15778.799805,15680.000000,15773.900391,378200,-0.073331
2,2021-06-09,15635.349609,15800.450195,15566.900391,15766.299805,457900,-0.665498
3,2021-06-10,15737.750000,15751.250000,15648.500000,15692.099609,298300,0.654929
4,2021-06-11,15799.349609,15835.549805,15749.799805,15796.450195,363000,0.391413


In [32]:
nifty["Month"] = nifty["Date"].dt.to_period("M")

nifty.head()

Price,Date,Close,High,Low,Open,Volume,Daily_Return,Month
Ticker,,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,,
0,2021-06-07,15751.650391,15773.450195,15678.099609,15725.099609,394000,NaN,2021-06
1,2021-06-08,15740.099609,15778.799805,15680.000000,15773.900391,378200,-0.073331,2021-06
2,2021-06-09,15635.349609,15800.450195,15566.900391,15766.299805,457900,-0.665498,2021-06
3,2021-06-10,15737.750000,15751.250000,15648.500000,15692.099609,298300,0.654929,2021-06
4,2021-06-11,15799.349609,15835.549805,15749.799805,15796.450195,363000,0.391413,2021-06


In [33]:
monthly_avg = nifty.groupby("Month")["Close"].mean()

monthly_avg.head()

Price,Close
Ticker,^NSEI
Month,
2021-06,15757.180501
2021-07,15783.097656
2021-08,16470.459449
2021-09,17508.614304
2021-10,18020.220020


In [34]:
monthly_return = nifty.groupby("Month")["Close"].apply(
    lambda x: ((x.iloc[-1] - x.iloc[0]) / x.iloc[0]) * 100
)

monthly_return.head()

Ticker,^NSEI
Month,
2021-06,-0.191411
2021-07,0.529654
2021-08,7.850406
2021-09,3.173416
2021-10,0.796254


In [35]:
monthly_volatility = nifty.groupby("Month")["Daily_Return"].std()

monthly_volatility.head()

Month
2021-06    0.448465
2021-07    0.568059
2021-08    0.572949
2021-09    0.605499
2021-10    0.823025
Freq: M, Name: Daily_Return, dtype: float64

In [36]:
def max_drawdown(series):
    peak = series.cummax()
    drawdown = ((series - peak) / peak) * 100
    return drawdown.min()

monthly_drawdown = nifty.groupby("Month")["Close"].apply(max_drawdown)

monthly_drawdown.head()

Ticker,^NSEI
Month,
2021-06,-1.171450
2021-07,-1.834319
2021-08,-0.987683
2021-09,-1.327067
2021-10,-4.358923


In [37]:
monthly_avg = monthly_avg.squeeze()
monthly_return = monthly_return.squeeze()
monthly_volatility = monthly_volatility.squeeze()
monthly_drawdown = monthly_drawdown.squeeze()

In [38]:
summary = pd.DataFrame({
    "Average_Close": monthly_avg,
    "Monthly_Return_%": monthly_return,
    "Monthly_Volatility": monthly_volatility,
    "Max_Drawdown_%": monthly_drawdown
})

summary.head()

,Average_Close,Monthly_Return_%,Monthly_Volatility,Max_Drawdown_%
Month,,,,
2021-06,15757.180501,-0.191411,0.448465,-1.171450
2021-07,15783.097656,0.529654,0.568059,-1.834319
2021-08,16470.459449,7.850406,0.572949,-0.987683
2021-09,17508.614304,3.173416,0.605499,-1.327067
2021-10,18020.220020,0.796254,0.823025,-4.358923


In [39]:
summary = summary.reset_index()

summary.head()

,Month,Average_Close,Monthly_Return_%,Monthly_Volatility,Max_Drawdown_%
0,2021-06,15757.180501,-0.191411,0.448465,-1.171450
1,2021-07,15783.097656,0.529654,0.568059,-1.834319
2,2021-08,16470.459449,7.850406,0.572949,-0.987683
3,2021-09,17508.614304,3.173416,0.605499,-1.327067
4,2021-10,18020.220020,0.796254,0.823025,-4.358923


In [40]:
summary.to_csv("monthly_nifty_summary.csv", index=False)

print("CSV Saved Successfully")

CSV Saved Successfully


In [41]:
summary.describe()

,Average_Close,Monthly_Return_%,Monthly_Volatility,Max_Drawdown_%
count,61.000000,61.000000,61.000000,61.000000
mean,20892.816129,0.501989,0.802025,-3.556543
std,3396.038630,3.772086,0.331372,2.250409
min,15757.180501,-10.191947,0.414672,-10.191947
25%,17710.679688,-2.369682,0.569806,-4.861264
50%,21165.990039,0.359728,0.705668,-3.312140
75%,23969.847451,3.209691,0.961849,-1.636149
max,25997.738725,8.927093,1.853947,-0.360225


In [42]:
summary.tail()

,Month,Average_Close,Monthly_Return_%,Monthly_Volatility,Max_Drawdown_%
56,2026-02,25625.575098,0.359728,0.930276,-2.986837
57,2026-03,23593.484272,-10.191947,1.609402,-10.191947
58,2026-04,23879.292480,5.812104,1.195700,-2.761368
59,2026-05,23822.536801,-2.369682,0.866937,-3.910240
60,2026-06,23411.000000,-0.068001,0.423709,-0.497589
